In [1]:
import torch 
from unsloth import FastLanguageModel
from trl import SFTTrainer 
from transformers import TrainingArguments 
from unsloth.chat_templates import get_chat_template , standardize_sharegpt


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


c:\Users\karan\PycharmProjects\ragvsfinetuning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
import os
os.environ["PYTHONIOENCODING"] = "utf-8"


In [3]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.backends.cudnn.enabled)

2.7.1+cu126
12.6
True


In [4]:
model , tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit" , 
    max_seq_length = 2048 , 
    dtype = None ,
    load_in_4bit = True
)

c:\Users\karan\PycharmProjects\ragvsfinetuning\venv\lib\site-packages\unsloth_zoo\gradient_checkpointing.py:339: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  GPU_BUFFERS = tuple([torch.empty(2*256*2048, dtype = dtype, device = f"{DEVICE_TYPE}:{i}") for i in range(n_gpus)])


==((====))==  Unsloth 2025.8.1: Fast Llama patching. Transformers: 4.54.1.
   \\   /|    NVIDIA GeForce RTX 3050 Laptop GPU. Num GPUs = 1. Max memory: 4.0 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [5]:
model = FastLanguageModel.get_peft_model(
    model , r = 16 , 
    target_modules=["q_proj" , "k_proj" , "v_proj" , "o_proj" , "gate_proj" , "up_proj" , "down_proj"]
)

Unsloth 2025.8.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
tokenizer = get_chat_template(
   tokenizer,
   chat_template = "llama-3.2",
)

tokenizer.pad_token = tokenizer.eos_token
FastLanguageModel.for_inference(model)

messages = [
   {"role": "user", "content": "What is the primary function of the product Damini of AtoZ company?"}
   ]

inputs = tokenizer.apply_chat_template(
   messages,
   tokenize=True,
   add_generation_prompt=True,
   return_tensors="pt",
   padding=True,  
).to("cuda")

attention_mask = inputs != tokenizer.pad_token_id

outputs = model.generate(
   input_ids=inputs,
   attention_mask=attention_mask,
   max_new_tokens=64,
   use_cache=True, 
   temperature=0.6,  
   min_p=0.1,
)

text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

user

What is the primary function of the product Damini of AtoZ company?assistant

I couldn't find any information about a product called "Damini" from an AtoZ company. Could you please provide more context or details about the product and company? I'll do my best to provide an accurate answer.


In [ ]:

print(f"Tokenizer vocab size: {tokenizer.vocab_size}")
print(f"Model vocab size: {model.config.vocab_size}")

if tokenizer.vocab_size != model.config.vocab_size:
    print("❌ TOKENIZER-MODEL MISMATCH!")
    print("You need to use the correct tokenizer for your model")

Tokenizer vocab size: 128000
Model vocab size: 128256
❌ TOKENIZER-MODEL MISMATCH!
You need to use the correct tokenizer for your model


In [8]:
from datasets import load_dataset

dataset = load_dataset("prsdm/Machine-Learning-QA-dataset" , split="train" , encoding="utf-8")


In [ ]:
def check_dataset_for_encoding_issues(dataset, fields=["Question", "Answer"]):
    import unicodedata

    bad_entries = []

    for i, example in enumerate(dataset):
        for field in fields:
            value = example.get(field, "Answer")
            try:
 
                encoded = value.encode("utf-8")
                decoded = encoded.decode("utf-8")
            except UnicodeDecodeError as e:
                print(f"❌ Decode error in example {i}, field '{field}': {e}")
                bad_entries.append((i, field, value))
            except UnicodeEncodeError as e:
                print(f"❌ Encode error in example {i}, field '{field}': {e}")
                bad_entries.append((i, field, value))

            for char in value:
                if unicodedata.category(char)[0] == "C" and char not in "\n\t":
                    print(f"⚠️ Control/non-printable char at index {i} in field '{field}': U+{ord(char):04X} ({repr(char)})")
                    bad_entries.append((i, field, value))
                    break  

    if bad_entries:
        print(f"\n⚠️ Found {len(bad_entries)} potentially problematic entries.")
    else:
        print("✅ No encoding or decoding issues found.")

    return bad_entries

In [11]:
bad_data = check_dataset_for_encoding_issues(dataset, fields=["Question", "Answer"])

✅ No encoding or decoding issues found.


In [12]:
def to_chat_format(example):
    return {
        "conversation": [
            {"role": "user", "content": example["Question"]},
            {"role": "assistant", "content": example["Answer"]},
        ]
    }

chat_data = list(map(to_chat_format, dataset))
print(chat_data)

[{'conversation': [{'role': 'user', 'content': 'What is the fundamental goal of machine learning?'}, {'role': 'assistant', 'content': 'The fundamental goal of machine learning is to develop algorithms that enable computers to learn from data, recognize patterns, and make intelligent decisions or predictions without explicit programming.'}]}, {'conversation': [{'role': 'user', 'content': 'Explain the concept of overfitting in machine learning.'}, {'role': 'assistant', 'content': 'Overfitting occurs when a machine learning model learns the training data too well, capturing noise or random fluctuations. As a result, the model may perform poorly on new, unseen data because it has essentially memorized the training set.'}]}, {'conversation': [{'role': 'user', 'content': 'Can you distinguish between bias and variance in the context of machine learning?'}, {'role': 'assistant', 'content': "Bias refers to the error introduced by approximating a real-world problem, while variance is the amount 

In [13]:
from datasets import Dataset
train_dataset = Dataset.from_list(chat_data)
print(train_dataset)


Dataset({
    features: ['conversation'],
    num_rows: 101
})


In [14]:
type(train_dataset)

datasets.arrow_dataset.Dataset

In [ ]:
tokenizer = get_chat_template(
    tokenizer,  # Tokenizer being used
    chat_template="llama-3.2",  # The chat template format
)

def formatting_prompts_func(examples):
    convos = examples["conversation"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)

Model does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


Map: 100%|██████████| 101/101 [00:00<00:00, 10090.15 examples/s]


In [ ]:
train_dataset['text']
print(train_dataset[0]["conversation"])
print(train_dataset[0]["text"])

[{'content': 'What is the fundamental goal of machine learning?', 'role': 'user'}, {'content': 'The fundamental goal of machine learning is to develop algorithms that enable computers to learn from data, recognize patterns, and make intelligent decisions or predictions without explicit programming.', 'role': 'assistant'}]
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the fundamental goal of machine learning?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

The fundamental goal of machine learning is to develop algorithms that enable computers to learn from data, recognize patterns, and make intelligent decisions or predictions without explicit programming.<|eot_id|>


In [61]:
import builtins
open = lambda file, mode='r', encoding='utf-8', *args, **kwargs: builtins.open(file, mode, encoding=encoding, *args, **kwargs)


In [17]:
import sys
print(sys.getdefaultencoding())

utf-8


In [19]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    return text.encode("utf-8", "replace").decode("utf-8", "replace")

def clean_dataset(dataset, fields=["text"]):
    def _clean(example):
        for f in fields:
            if f in example:
                example[f] = clean_text(example[f])
        return example
    return dataset.map(_clean)

train_dataset = clean_dataset(train_dataset, fields=["text"])
print(train_dataset)

Map: 100%|██████████| 101/101 [00:00<00:00, 10856.60 examples/s]

Dataset({
    features: ['conversation', 'text'],
    num_rows: 101
})


In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length= 2048,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=2,  
        gradient_accumulation_steps=4,  
        warmup_steps=5, 

        num_train_epochs=15,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",  
        report_to="none",  

    ),
)

Unsloth: Tokenizing ["text"]: 100%|██████████| 101/101 [00:00<00:00, 3606.73 examples/s]


In [ ]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",  
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n", 
)
trainer_stats = trainer.train()

Map (num_proc=8): 100%|██████████| 101/101 [00:14<00:00,  7.11 examples/s]
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 101 | Num Epochs = 15 | Total steps = 195
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,1.313800
2,1.527300
3,1.508700
4,1.522000
5,1.437200
6,1.291900
7,1.263200
8,1.200400
9,1.043300
10,1.213100


In [24]:
model.save_pretrained("finetuned_model")
tokenizer.save_pretrained("finetuned_model")

('finetuned_model\\tokenizer_config.json',
 'finetuned_model\\special_tokens_map.json',
 'finetuned_model\\chat_template.jinja',
 'finetuned_model\\tokenizer.json')